# BanglaSQL — Colab Training Notebook
**Natural Language (Bangla) to SQL — University Management System**

### Steps
Run each cell **in order**. Runtime → Change runtime type → **T4 GPU** before starting.

#Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1 — Check GPU

In [1]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


## Step 2 — Clone repository

In [2]:
# Replace with your actual GitHub repo URL
REPO_URL = 'https://github.com/mustafiz-07/BanglaSQL.git'

!git clone {REPO_URL} banglasql
%cd banglasql/
!ls -la

Cloning into 'banglasql'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 68 (delta 36), reused 49 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 64.94 KiB | 4.33 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/banglasql
total 168
drwxr-xr-x 4 root root  4096 Sep 13 13:44 .
drwxr-xr-x 1 root root  4096 Sep 13 13:44 ..
-rw-r--r-- 1 root root 25381 Sep 13 13:44 build_dataset.py
-rw-r--r-- 1 root root 46108 Sep 13 13:44 colab_train.ipynb
-rw-r--r-- 1 root root 14853 Sep 13 13:44 create_database.py
drwxr-xr-x 2 root root  4096 Sep 13 13:44 data
-rw-r--r-- 1 root root   232 Sep 13 13:44 docker-compose.yml
-rw-r--r-- 1 root root   360 Sep 13 13:44 Dockerfile
-rw-r--r-- 1 root root  2331 Sep 13 13:44 er_diagram.md
drwxr-xr-x 8 root root  4096 Sep 13 13:44 .git
-rw-r--r-- 1 root root   572 Sep 13 13:44 .gitignore
-rw-r--r-- 1 root root 11360 Sep 13 13:44 prepro

## Step 3 — Install dependencies

In [3]:
# NOTE: train.py's own Quick Start instructions call for
# requirements_colab.txt (a Colab-specific pin list — Colab already ships a
# working torch/CUDA build, so this avoids reinstalling a conflicting one).
# This cell previously referenced a plain 'requirements.txt', which doesn't
# match and would fail with 'file not found' on a fresh clone.
!pip install -r requirements_colab.txt -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.8 MB/s eta 0:00:00
Dependencies installed.


## Step 4 — Generate database & dataset

In [4]:
!python create_database.py

Creating schema in: /content/banglasql/banglasql.db
Populating with synthetic data...

BanglaSQL Database — Summary
  departments     :    10 rows
  instructors     :    50 rows
  students        :   304 rows
  courses         :    80 rows
  enrollments     :   471 rows
  attendance      :   500 rows

-- Sample: Top 5 students by CGPA --
  Tasnim Sultana — CGPA: 4.0
  Farhana Rahman — CGPA: 3.97
  Nazmul Alam — CGPA: 3.96
  Joynal Mondal — CGPA: 3.96
  Tasnim Begum — CGPA: 3.96

-- Sample: Students per department --
  Mechanical Engineering                        : 34 students
  Physics                                       : 34 students
  Computer Science and Engineering              : 33 students
  Business Administration                       : 32 students
  Civil Engineering                             : 32 students
  Mathematics                                   : 30 students
  Electrical and Electronic Engineering         : 29 students
  English                                   

In [5]:
!python build_dataset.py

BanglaSQL Dataset Builder — Phase 2

[1/5] Loaded 165 base templates
      Easy: 65
      Medium: 100

[2/5] After augmentation: 2475 pairs (before deduplication)

[3/5] After deduplication: 2475 pairs
      Easy:   975
      Medium: 1500

  [OK] Dataset size (2475 pairs) is within target range.

[4/5] Split:
      Train : 1381 (56%)
      Dev   : 577   (23%)
      Test  : 517  (21%)
      (Test patterns held out entirely from train for generalization)

[5/5] Saved /content/banglasql/data/dataset_train.json

[5/5] Saved /content/banglasql/data/dataset_dev.json

[5/5] Saved /content/banglasql/data/dataset_test.json

      Saved stats: /content/banglasql/data/dataset_stats.json

Dataset Stats Summary
{
  "base_templates": {
    "easy": 65,
    "medium": 100,
    "total": 165
  },
  "after_augmentation_dedup": {
    "easy": 975,
    "medium": 1500,
    "total": 2475
  },
  "splits": {
    "train": 1381,
    "dev": 577,
    "test": 517
  },
  "train_easy": 451,
  "train_medium": 930,
  "de

## Step 5 — Tokenizer analysis & preprocessing

In [6]:
!python preprocess_check.py

BanglaSQL — Phase 3: Preprocessing & Tokenizer Analysis

[1/3] Unicode Normalization (NFC)...
  train: 1381 pairs, 0 normalized
  dev: 577 pairs, 0 normalized
  test: 517 pairs, 0 normalized

  (Analysis below uses train+dev only — 1958 pairs. Test split (517 pairs) is excluded from here on so hyperparameter choices can't leak information from it.)

[2/3] Tokenizer Analysis...

  Loading tokenizer: csebuetnlp/banglat5
config.json: 100% 659/659 [00:00<00:00, 2.83MB/s]
tokenizer_config.json: 100% 1.83k/1.83k [00:00<00:00, 5.13MB/s]

spiece.model: downloading bytes:   3% 33.2k/1.11M [00:01<00:43, 24.7kB/s]
spiece.model: downloading bytes: 100% 697k/697k [00:01<00:00, 493kB/s, 67.1kB/s  ] 
spiece.model: reconstructing file: 100% 1.11M/1.11M [00:01<00:00, 786kB/s,  107kB/s  ]
special_tokens_map.json: 100% 1.79k/1.79k [00:00<00:00, 5.31MB/s]

  Model: csebuetnlp/banglat5
  Vocab size: 32,100
  Bangla-script tokens in vocab: 28,644 (89.2%)
  Avg tokens per question (sample of 50): 13.8
  Avg 

## Step 6 — Train
> Expected time: ~60–90 min on a T4 for a full 20-epoch run over ~1380+
> training pairs (exact split sizes are printed by build_dataset.py).
> Early stopping monitors eval_exact_match with patience=5 and warm-up=6 epochs,
> and the highest exact_match checkpoint is restored and saved to checkpoints/best_model.
> Dev evaluation uses beam search (num_beams=4) matching inference.

In [7]:
!python train.py

2026-09-13 13:45:47,470 [INFO] Loaded train config from /content/banglasql/data/train_config.json
2026-09-13 13:45:47,471 [INFO] ============================================================
2026-09-13 13:45:47,471 [INFO] BanglaSQL — Phase 3: Training
2026-09-13 13:45:47,471 [INFO] ============================================================
2026-09-13 13:45:47,471 [INFO] Model       : csebuetnlp/banglat5
2026-09-13 13:45:47,471 [INFO] Max input   : 256
2026-09-13 13:45:47,471 [INFO] Max target  : 128
2026-09-13 13:45:47,471 [INFO] Batch size  : 8 (x2 grad accum = 16 effective)
2026-09-13 13:45:47,471 [INFO] Epochs      : 20
2026-09-13 13:45:47,471 [INFO] LR          : 0.0002
2026-09-13 13:45:47,471 [INFO] Early stop  : patience=5, warm-up=6 epochs
2026-09-13 13:45:47,471 [INFO] Device      : cuda
2026-09-13 13:45:47,506 [INFO] GPU Name    : Tesla T4
2026-09-13 13:45:47,506 [INFO] 
Loading tokenizer & model: csebuetnlp/banglat5
2026-09-13 13:45:47,843 [INFO] HTTP Request: HEAD https://h

## Step 7 — Save model to Google Drive (prevents loss on session timeout)

In [8]:


import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/BanglaSQL/checkpoints3'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy the best model checkpoint
src = 'checkpoints/best_model'
dst = os.path.join(DRIVE_SAVE_DIR, 'best_model')

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Best model saved to Google Drive: {dst}')
else:
    print('No best_model found. Check that training completed successfully.')

ValueError: mount failed

## Step 8 — Quick inference test (verify the model works)

In [ ]:
import json, unicodedata
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load config
with open('data/train_config.json') as f:
    cfg = json.load(f)

MODEL_DIR    = 'checkpoints/best_model'
SCHEMA       = cfg['schema_string']
MAX_IN       = cfg['max_input_length']
MAX_OUT      = cfg['max_target_length']

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

def predict(bangla_question: str) -> str:
    q   = unicodedata.normalize('NFC', bangla_question.strip())
    inp = f'translate Bangla to SQL: {q} </s> {SCHEMA}'
    ids = tokenizer(inp, return_tensors='pt', max_length=MAX_IN, truncation=True)
    out = model.generate(**ids, max_length=MAX_OUT, num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Test questions
test_questions = [
    'সকল শিক্ষার্থীর তালিকা দাও।',
    'যেসব শিক্ষার্থীর CGPA ৩.৫-এর বেশি তাদের নাম দাও।',
    'প্রতিটি বিভাগে কতজন শিক্ষার্থী আছে তা দেখাও।',
]

print('=== Inference Test ===\n')
for q in test_questions:
    sql = predict(q)
    print(f'Q: {q}')
    print(f'SQL: {sql}')
    print()

## Step 9 — Download model (alternative to Drive)
If you prefer to download the checkpoint directly:

In [ ]:
import shutil
shutil.make_archive('best_model', 'zip', 'checkpoints/best_model')

from google.colab import files
files.download('best_model.zip')